In [1]:
import os, sys
os.chdir(os.path.expanduser('~/QIAO0042/models/acv/facemask/'))
sys.path.insert(0, os.getcwd())
print('CWD:', os.getcwd())

CWD: /scratch-share/QIAO0042/models/acv/facemask


In [2]:
import os, time, logging, shutil
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from pathlib import Path
from PIL import Image as PILImage
from contextlib import contextmanager
import albumentations as A
import cv2
import wandb

from unet import LiteUNetV2
from losses import FocalDiceLoss
from metrics import confusion_matrix, f1_macro_from_cm
from palette import NUM_CLASSES, rgb_to_label
from dataset import FaceParsingDataset
from split_utils import list_images, make_split
from augment import flip_with_label_swap

logging.getLogger('torch._inductor').setLevel(logging.WARNING)
logging.getLogger('torch._dynamo').setLevel(logging.WARNING)

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_bf16  = device.type == 'cuda' and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
torch.backends.cudnn.benchmark = True
print(f'Device: {device}  AMP: {amp_dtype}')

Device: cuda  AMP: torch.bfloat16


In [3]:
# -----------------------------------------------------------------------
# Experiment config
# -----------------------------------------------------------------------
SEED        = 42
VAL_RATIO   = 0.1
EPOCHS      = 50
BATCH       = 16
LR          = 2e-2
WARMUP      = 5
AUX_W       = 0.4
OHEM_RATIO  = 0.7
EMA_DECAY   = 0.999
CKPT_PATH   = 'checkpoints/liteunetv2_best.pt'

# ImageNet channel normalization applied at batch level
# Images come out of the dataset as float32 in [0, 1]
NORM_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
NORM_STD  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def normalize(imgs):
    """Apply ImageNet mean/std normalization to a batch of [0,1] float tensors."""
    return (imgs - NORM_MEAN) / NORM_STD

print(f'Epochs: {EPOCHS}  Batch: {BATCH}  LR: {LR}')
print(f'Normalization: mean={NORM_MEAN.squeeze().tolist()}  std={NORM_STD.squeeze().tolist()}')

Epochs: 50  Batch: 16  LR: 0.02
Normalization: mean=[0.48500001430511475, 0.4560000002384186, 0.4059999883174896]  std=[0.2290000021457672, 0.2240000069141388, 0.22499999403953552]


In [4]:
# -----------------------------------------------------------------------
# Stronger augmentation
# Changes vs baseline:
#   - ColorJitter: brightness/contrast 0.3→0.5, saturation 0.15→0.3, hue 0.05→0.10
#   - GaussianBlur: p 0.15→0.25
#   - Added GridDistortion (mild, p=0.3) for boundary robustness
#   - Added GaussNoise (p=0.25) to simulate sensor noise
# -----------------------------------------------------------------------
_FLIP_PAIRS = [(4, 5), (6, 7), (8, 9)]   # l/r eye, brow, ear

_album_strong = A.Compose([
    A.ShiftScaleRotate(
        shift_limit=0.03, scale_limit=0.20, rotate_limit=10,
        border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0, p=0.7,
    ),
    A.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.3, hue=0.10, p=0.8),
    A.GaussianBlur(blur_limit=(3, 5), p=0.25),
    A.GridDistortion(num_steps=5, distort_limit=0.15,
                     border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0, p=0.3),
    A.GaussNoise(std_range=(0.01, 0.05), p=0.25),
])

def aug_strong(img: np.ndarray, mask: np.ndarray):
    img, mask = flip_with_label_swap(img, mask, p=0.5)
    out = _album_strong(image=img, mask=mask.astype(np.int32))
    return out['image'], out['mask'].astype(np.int64)

print('Strong augmentation pipeline ready.')

Strong augmentation pipeline ready.


/home/msai/qiao0042/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_350366/1672337053.py:12: UserWarning: Argument(s) 'value, mask_value' are not valid for transform ShiftScaleRotate
  A.ShiftScaleRotate(
/tmp/ipykernel_350366/1672337053.py:18: UserWarning: Argument(s) 'value, mask_value' are not valid for transform GridDistortion
  A.GridDistortion(num_steps=5, distort_limit=0.15,


In [5]:
# -----------------------------------------------------------------------
# Data — copy to /tmp, split, class weights
# -----------------------------------------------------------------------
IMG_DIR  = '/tmp/facemask/images'
MASK_DIR = '/tmp/facemask/masks'

for src, dst in [('train/images', IMG_DIR), ('train/masks', MASK_DIR)]:
    if not Path(dst).exists():
        shutil.copytree(Path(src).resolve(), dst)
        print(f'  copied {src} -> {dst}')
    else:
        print(f'  already exists: {dst}')

all_files = list_images(IMG_DIR)
train_files, val_files = make_split(all_files, val_ratio=VAL_RATIO, seed=SEED)
print(f'Split: {len(train_files)} train / {len(val_files)} val')

# Class weights (sqrt median-frequency)
pixel_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
mask_lookup  = {p.stem: p for p in Path(MASK_DIR).iterdir()
                if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}}
for fn in train_files:
    labels = rgb_to_label(np.array(PILImage.open(mask_lookup[Path(fn).stem]).convert('RGB'), dtype=np.uint8))
    for c in range(NUM_CLASSES):
        pixel_counts[c] += int((labels == c).sum())

freq             = pixel_counts / pixel_counts.sum()
median_freq      = float(np.median(freq[freq > 0]))
class_weights_np = np.where(freq > 0, np.sqrt(median_freq / freq), 1.0)
CLASS_WEIGHTS    = torch.tensor(class_weights_np, dtype=torch.float32).to(device)

train_ds = FaceParsingDataset(IMG_DIR, MASK_DIR, file_list=train_files, augment=aug_strong, cache=True)
val_ds   = FaceParsingDataset(IMG_DIR, MASK_DIR, file_list=val_files,   augment=None,       cache=True)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_ds,   batch_size=8,     shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2)

print('Data preparation done.')

  already exists: /tmp/facemask/images
  already exists: /tmp/facemask/masks
Split: 900 train / 100 val


  → cached 900 decoded arrays in RAM
Loaded 900 samples from /tmp/facemask/images
  → cached 100 decoded arrays in RAM
Loaded 100 samples from /tmp/facemask/images
Data preparation done.


In [6]:
# -----------------------------------------------------------------------
# Model, loss, optimizer
# -----------------------------------------------------------------------
model = LiteUNetV2(num_classes=NUM_CLASSES, base=32, dropout=0.3,
                   deep_supervision=True).to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

criterion = FocalDiceLoss(NUM_CLASSES, dice_weight=0.85, gamma=2.0,
                          class_weights=CLASS_WEIGHTS, ohem_ratio=OHEM_RATIO)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        torch.optim.lr_scheduler.LinearLR(optimizer, 0.1, 1.0, total_iters=WARMUP),
        torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP, eta_min=1e-6),
    ],
    milestones=[WARMUP],
)
scaler = torch.amp.GradScaler('cuda', enabled=(not use_bf16 and device.type == 'cuda'))

print(f'Params: {num_params:,}')
print(f'Steps/epoch: {len(train_loader)}')

Params: 500,112
Steps/epoch: 57


In [7]:
# -----------------------------------------------------------------------
# EMA + validate
# -----------------------------------------------------------------------
class EMAKeeper:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.num_updates = 0
        self.shadow = {n: p.data.clone() for n, p in model.named_parameters() if p.requires_grad}

    @torch.no_grad()
    def update(self, model):
        self.num_updates += 1
        d = min(self.decay, (1 + self.num_updates) / (10 + self.num_updates))
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(d).add_(p.data, alpha=1 - d)

    @contextmanager
    def applied(self, model):
        original = {n: p.data.clone() for n, p in model.named_parameters() if p.requires_grad}
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.shadow[n])
        try:
            yield
        finally:
            for n, p in model.named_parameters():
                if p.requires_grad:
                    p.data.copy_(original[n])


def _downsample_mask(masks, size):
    return F.interpolate(masks.float().unsqueeze(1), size=size, mode='nearest').squeeze(1).long()


@torch.no_grad()
def validate(model, loader, ema=None):
    ctx = ema.applied(model) if ema is not None else contextmanager(lambda: (yield))()
    model.eval()
    cm_total = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.int64)
    with ctx:
        for imgs, masks, _ in loader:
            imgs  = normalize(imgs.to(device, non_blocking=True))
            masks = masks.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=device.type == 'cuda'):
                logits      = model(imgs)
                logits_flip = model(imgs.flip(-1)).flip(-1)
                for a, b in _FLIP_PAIRS:
                    logits_flip[:, [a, b]] = logits_flip[:, [b, a]]
                logits = (logits + logits_flip) * 0.5
            cm_total += confusion_matrix(logits.argmax(1).cpu(), masks.cpu(), num_classes=NUM_CLASSES)
    return f1_macro_from_cm(cm_total)


print('Training helpers ready.')

Training helpers ready.


In [8]:
# -----------------------------------------------------------------------
# Training
# -----------------------------------------------------------------------
wandb.init(
    project='face-parsing-unet',
    name=f'liteunetv2_base32_strongAug_e{EPOCHS}',
    config={
        'model': 'LiteUNetV2',
        'base': 32,
        'epochs': EPOCHS, 'batch': BATCH, 'lr': LR, 'warmup': WARMUP,
        'ohem_ratio': OHEM_RATIO, 'ema_decay': EMA_DECAY,
        'imagenet_norm': True,
        'aug': 'flip+ShiftScaleRotate+StrongColorJitter+GridDistortion+GaussNoise+Blur',
        'deep_supervision': True,
    },
)

os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
ema     = EMAKeeper(model, decay=EMA_DECAY)
best_f1 = -1.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    running = 0.0

    for imgs, masks, _ in train_loader:
        imgs  = normalize(imgs.to(device, non_blocking=True))
        masks = masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=device.type == 'cuda'):
            outputs = model(imgs)
            if isinstance(outputs, tuple):
                main, aux1, aux2 = outputs
                loss = (criterion(main, masks)
                        + AUX_W * criterion(aux1, _downsample_mask(masks, aux1.shape[-2:]))
                        + AUX_W * criterion(aux2, _downsample_mask(masks, aux2.shape[-2:])))
            else:
                loss = criterion(outputs, masks)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        ema.update(model)
        running += loss.item()

    scheduler.step()
    train_loss = running / len(train_loader)
    val_f1, _  = validate(model, val_loader, ema=ema)
    elapsed    = time.time() - t0

    wandb.log({'epoch': epoch, 'train/loss': train_loss,
               'val/macro_f1': val_f1, 'lr': scheduler.get_last_lr()[0]})
    print(f'[{epoch:03d}/{EPOCHS}] loss={train_loss:.4f}  val_F1={val_f1:.4f}  '
          f'lr={scheduler.get_last_lr()[0]:.2e}  ({elapsed:.1f}s)')

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save({'model': model.state_dict(), 'ema_shadow': ema.shadow,
                    'epoch': epoch, 'best_f1': best_f1}, CKPT_PATH)
        print(f'  saved {CKPT_PATH}  (F1={best_f1:.4f})')

print(f'\nBest val macro-F1: {best_f1:.4f}')
wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/msai/qiao0042/.netrc.
wandb: Currently logged in as: 584832452 (584832452-nanyang-technological-university-singapore) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[001/50] loss=1.9554  val_F1=0.3487  lr=5.60e-03  (18.7s)
  saved checkpoints/liteunetv2_best.pt  (F1=0.3487)
[002/50] loss=1.1496  val_F1=0.6046  lr=9.20e-03  (14.1s)
  saved checkpoints/liteunetv2_best.pt  (F1=0.6046)
[003/50] loss=0.9447  val_F1=0.6630  lr=1.28e-02  (14.3s)
  saved checkpoints/liteunetv2_best.pt  (F1=0.6630)
[004/50] loss=0.8817  val_F1=0.6665  lr=1.64e-02  (14.2s)
  saved checkpoints/liteunetv2_best.pt  (F1=0.6665)


/home/msai/qiao0042/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


[005/50] loss=0.8438  val_F1=0.6793  lr=2.00e-02  (14.4s)
  saved checkpoints/liteunetv2_best.pt  (F1=0.6793)
[006/50] loss=0.8172  val_F1=0.6488  lr=2.00e-02  (14.2s)
[007/50] loss=0.8059  val_F1=0.6803  lr=1.99e-02  (14.4s)
  saved checkpoints/liteunetv2_best.pt  (F1=0.6803)
[008/50] loss=0.7441  val_F1=0.7150  lr=1.98e-02  (14.2s)
  saved checkpoints/liteunetv2_best.pt  (F1=0.7150)
[009/50] loss=0.7204  val_F1=0.7245  lr=1.96e-02  (14.4s)
  saved checkpoints/liteunetv2_best.pt  (F1=0.7245)
[010/50] loss=0.7085  val_F1=0.7395  lr=1.94e-02  (14.3s)
  saved checkpoints/liteunetv2_best.pt  (F1=0.7395)
[011/50] loss=0.6818  val_F1=0.7407  lr=1.91e-02  (14.4s)
  saved checkpoints/liteunetv2_best.pt  (F1=0.7407)
[012/50] loss=0.6510  val_F1=0.7436  lr=1.88e-02  (14.4s)
  saved checkpoints/liteunetv2_best.pt  (F1=0.7436)
[013/50] loss=0.6595  val_F1=0.7278  lr=1.85e-02  (14.3s)
[014/50] loss=0.6468  val_F1=0.7601  lr=1.81e-02  (14.3s)
  saved checkpoints/liteunetv2_best.pt  (F1=0.7601)
[015

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
lr,▃▄▅▇██████▇▇▇▇▇▆▆▆▆▆▅▅▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁▁▁▁
train/loss,█▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/macro_f1,▁▅▆▆▆▆▇▇▇▇▇█▇▇▇▇▇███████████████████████
epoch,50
lr,0.0
train/loss,0.40849
val/macro_f1,0.78846
